Imports

In [1]:
%load_ext autoreload
%autoreload 2
from tqdm import tqdm
import random
import numpy as np
from open3dsg.open_dataset import Open2D3DSGDataset
import json
from llava.model import *
import torch
from llava.model.builder import load_pretrained_model
from constants import get_paths
from llava.mm_utils import (
    tokenizer_special_token,
    process_videos
)

/home/alegretelena/miniconda3/envs/llava-3d/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Paths (if drive is mounted, else leave them out e.g. the cache dir)

blip_cache_dir = '/content/drive/MyDrive/master_practical/models/BLIP'

llava_cache_dir = '/content/drive/MyDrive/master_practical/models/LLaVA3D'

path_3rscan_raw = '/content/drive/MyDrive/master_practical/data/3rscan'

path_3rscan_metadata = './data/3rscan'

path_3rscan_processed = './data/3rscan/processed'


In [13]:
llava_cache_dir, path_3rscan_raw, path_3rscan_metadata, path_3rscan_processed = get_paths("Elena")

If you have spare disk on your drive, optionally download models here once


In [ ]:
#InstructBlipVideoForConditionalGeneration.from_pretrained("Salesforce/instructblip-vicuna-7b", cache_dir=blip_cache_dir)
#LlavaLlamaForCausalLM.from_pretrained('ChaimZhu/LLaVA-3D-7B',cache_dir=llava_cache_dir)

### Data Loading



In [14]:
## Read 10 training samples
import os
with open('/mnt/scratch/LLaVA-3D/data/train_scans.txt', 'r') as file:
    scans_train = [line.strip() for line in file]

mode = 'train'
relationships = json.load(open(os.path.join(path_3rscan_metadata, f"relationships_{mode}.json")))["scans"]
relationship_list = []
for r in relationships:
    if r['scan'] in scans_train and r['split'] == 1:
        relationship_list.append(r)

# Dataset of length 10
dataset = Open2D3DSGDataset(
    relationships_R3SCAN=relationship_list,
    relationships_scannet=None,
    path_3rscan_raw = path_3rscan_raw,
    path_3rscan_processed = path_3rscan_processed,
    openseg=False,
    img_dim=224,
    rel_img_dim=224,
    top_k_frames=5,
    scales=3,
    mini=False,
    load_features=None,
    blip=True,
    llava=False,
    half=False,
    max_objects=9,
    max_rels=72
)

  0%|          | 0/15 [00:00<?, ?it/s]

[Errno 2] No such file or directory: '/mnt/scratch/LLaVA-3D/data/3rscan/processed/preprocessed/4acaebc0-6c10-2a2a-852e-0226d6539299/data_dict_1.pkl'[Errno 2] No such file or directory: '/mnt/scratch/LLaVA-3D/data/3rscan/processed/preprocessed/4fbad32b-465b-2a5d-8499-85100e88f454/data_dict_1.pkl'[Errno 2] No such file or directory: '/mnt/scratch/LLaVA-3D/data/3rscan/processed/preprocessed/752cc5a3-920c-26f5-8ff3-49518eff94c6/data_dict_1.pkl' [Errno 2] No such file or directory: '/mnt/scratch/LLaVA-3D/data/3rscan/processed/preprocessed/f3d7fa58-2835-2805-83bc-d2c583045bb4/data_dict_1.pkl'/mnt/scratch/LLaVA-3D/data/3rscan/processed/preprocessed/4acaebc0-6c10-2a2a-852e-0226d6539299/data_dict_1.pkl   
/mnt/scratch/LLaVA-3D/data/3rscan/processed/preprocessed/4fbad32b-465b-2a5d-8499-85100e88f454/data_dict_1.pkl[Errno 2] No such file or directory: '/mnt/scratch/LLaVA-3D/data/3rscan/processed/preprocessed/198aaa76-0ba3-26f6-84de-6c13263a60bc/data_dict_1.pkl'/mnt/scratch/LLaVA-3D/data/3rscan/pro

100%|██████████| 15/15 [00:00<00:00, 1207.80it/s]

  /mnt/scratch/LLaVA-3D/data/3rscan/processed/preprocessed/531cff08-0021-28f6-8e08-ba2eeb945e09/data_dict_1.pkl/mnt/scratch/LLaVA-3D/data/3rscan/processed/preprocessed/0cf75f50-564d-23f5-8a6b-cf1f98afcbce/data_dict_1.pkl/mnt/scratch/LLaVA-3D/data/3rscan/processed/preprocessed/5ed77dd4-c4f5-27a0-8476-f3048ea53ef5/data_dict_1.pkl




### Generate Queries and helper methods

In [15]:
def get_queries(data_dict):

  obj_class_dict = [line.rstrip() for line in open(os.path.join(path_3rscan_metadata, "classes.txt"), "r").readlines()]
  bidx = 7
  obj_count = data_dict['objects_count'].item()
  rel_count = int(data_dict["predicate_count"].item())
  objects_gt = data_dict['objects_cat']
  edges = data_dict['edges'][:rel_count]
  object_edges = np.array(objects_gt[:obj_count][edges], dtype=np.int32)
  object_edges = np.array(obj_class_dict)[object_edges]
  object_edges[object_edges == 'socket'] = 'wall'
  queries = [
  f"Describe the relationship between the {o[0]} and the {'other ' if o[0]==o[1] else ''}{o[1]}. Start the response with: the {o[0]}" if o[0] != o[1]
  else f"Describe the relationship between the {o[0]} and the {'other ' if o[0]==o[1] else ''}{o[1]}. Start the response with: the {o[0]}"
  for o in object_edges]

  return queries

def is_black_image(image_array):
    """
    Check if a given image array is completely black.
    A completely black image has all pixel values equal to 0.
    """
    return np.all(image_array == 0)

def process_images(rel_images):
  """
  processes images for blip model (takes 4 images as input)
  """
  image_shape = (960, 540, 3)
  black_image = np.zeros(image_shape, dtype=np.uint8)
  processed_images = []

  for image_list in rel_images:
      # Pad with black images if the list has fewer than 4 images
      if len(image_list) < 4:
          padded_list = image_list + [black_image] * (4 - len(image_list))
      else:
          padded_list = image_list

      # Select a random subset of 4 images
      random_subset = [np.array(x) for x in random.sample(padded_list, 4)]

      # Check if all images in the subset are black
      if all(is_black_image(image) for image in random_subset):
          processed_images.append(np.empty((0,)))  # Append an empty array
      else:
          # Stack the subset and append it to the result
          processed_images.append(np.stack(random_subset))

  return processed_images

### LLava3D

Init the model

In [16]:
model_path = "ChaimZhu/LLaVA-3D-7B"
model_name = "LLaVA-3D-7B"
tokenizer, model, processor, context_len = load_pretrained_model(
    model_path=model_path, model_base=None, model_name=model_name, torch_dtype=torch.float16, load_4bit=True, cache_dir=llava_cache_dir
)

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/master_practical/data/embodiedscan_infos_full.json'

In [30]:
def sanity_check(videos_dict):

  import matplotlib.pyplot as plt

  # Extraer 5 imágenes al inicio y 5 al final
  fig, axes = plt.subplots(2, 5, figsize=(15, 6))

  print('Sanity Check images: ')
  print('     images_tensor: ', type(videos_dict['images']), videos_dict['images'][0].shape)
  import torch

  images_tensor = videos_dict['images'][0]  # Extraer el batch (B=1)

  # Crear una lista de diferencias cada 5 frames
  differences = [torch.abs(images_tensor[i + 5] - images_tensor[i]).sum() for i in range(15)]

  # Convertir a tensor para facilitar la visualización
  differences = torch.tensor(differences)

  print(f"[DEBUG] - Diferencias entre frames cada 5 pasos: {differences}")


  for i in range(5):
    img1 = videos_dict['images'][0, i].permute(1, 2, 0).cpu().numpy()  # Convertir a formato (H, W, C)
    img2 = videos_dict['images'][0, -5 + i].permute(1, 2, 0).cpu().numpy()

    axes[0, i].imshow(img1)
    axes[0, i].axis("off")
    axes[0, i].set_title(f"Frame {i+1}")

    axes[1, i].imshow(img2)
    axes[1, i].axis("off")
    axes[1, i].set_title(f"Frame {20 - 5 + i + 1}")

  plt.show()


Loop through scenes

In [ ]:
for data_dict in tqdm(dataset):
  scan_id = data_dict['scene_id']
  video_path = os.path.join(path_3rscan_raw, scan_id)
  queries = get_queries(data_dict)
  results = []
  file_name = f"results_{scan_id}_llava.json"
  ## ADJUST!
  path = '/content/drive/MyDrive/4tCarrera/master_practical/results/quantization/4bit'
  results_path = os.path.join(path, file_name)

  ## Loop through relationships/queries
  for idx, prompt in enumerate(queries):
    videos_dict = process_videos(
      video_path,
      processor['video'],
      mode='random',
      device=model.device,
      text="hellp",
      data_dict=data_dict,
      use_relationship=idx,
      balance_img_with='doubling_img',
      )

    clicks = torch.zeros((0,3))

    sanity_check(videos_dict)

    stop

    images_tensor = videos_dict['images'].to(device=model.device, dtype=model.dtype) # Shape: [B, num_frames, channels, H, W]=[1, 20, 3, 336, 336]
    depths_tensor = videos_dict['depths'].to(device=model.device, dtype=model.dtype)
    poses_tensor = videos_dict['poses'].to(device=model.device, dtype=model.dtype)
    intrinsics_tensor = videos_dict['intrinsics'].to(device=model.device, dtype=model.dtype)
    clicks_tensor = clicks.to(model.device, dtype=torch.bfloat16)

    input_ids = (
      tokenizer_special_token(prompt, tokenizer, return_tensors="pt")
      .unsqueeze(0)
      .cuda()
    )

    # No relationships
    if torch.all(images_tensor == 0):
      results.append({"query": prompt, "output": "No relationship"})
      with open(results_path, "w") as json_file:
        json.dump(results, json_file, indent=4)
      continue

    with torch.inference_mode():
        output_ids = model.generate(
            input_ids,
            images=images_tensor,
            depths=depths_tensor,
            poses=poses_tensor,
            intrinsics=intrinsics_tensor,
            clicks=clicks_tensor,
            image_sizes=None,
            do_sample=True,
            temperature=0.2,
            top_p=None,
            num_beams=1,
            max_new_tokens=512,
            use_cache=True,
        )
    print('DONEEEEE!!!!!!!!')
    outputs = tokenizer.batch_decode(output_ids, skip_special_tokens=True)[0].strip()
    ## Save prompt + result as json
    results.append({"query": prompt, "output": outputs})
    with open(results_path, "w") as json_file:
        json.dump(results, json_file, indent=4)

    print('DONEEEEE2222222222!!!!!!!!')

### BLIP

Init the model

In [ ]:
kwargs = {"device_map": 0}
kwargs['quantization_config'] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type='nf4'
        )
processor = InstructBlipVideoProcessor.from_pretrained("Salesforce/instructblip-vicuna-7b")
model_blip = InstructBlipVideoForConditionalGeneration.from_pretrained("Salesforce/instructblip-vicuna-7b", cache_dir=blip_cache_dir, **kwargs)

In [ ]:
data_dict = dataset[0]

In [ ]:
for data_dict in tqdm(dataset):
  print(data_dict['objects_id'])

Loop through scenes

In [ ]:
for data_dict in tqdm(dataset):
  scan_id = data_dict['scene_id']
  queries = get_queries(data_dict)
  results = []
  file_name = f"results_{scan_id}_blip.json"
  ## ADJUST!
  path = '/content/drive/MyDrive/master_practical/results/quantization/4bit'
  results_path = os.path.join(path, file_name)

  processed_images = process_images(data_dict['blip_images'])

  for idx, prompt in enumerate(queries):
    image_set = processed_images[idx]

    #No relationships
    if np.all(image_set == 0):
      results.append({"query": prompt, "output": "No relationship"})
      with open(results_path, "w") as json_file:
        json.dump(results, json_file, indent=4)
      continue
    inputs = processor(text=prompt, images=image_set, return_tensors="pt").to(model_blip.device)
    outputs = model_blip.generate(
        **inputs,
        do_sample=False,
        num_beams=5,
        repetition_penalty=1.5,
        length_penalty=1.0,
        max_new_tokens=30
    )
    generated_text = processor.batch_decode(outputs, skip_special_tokens=True)[0].strip()
    results.append({"query": prompt, "output": generated_text})

    ## Save prompt + result as json
    with open(results_path, "w") as json_file:
        json.dump(results, json_file, indent=4)